# From Many Experts to One
## A hands-on tour of agglomerative multi-teacher knowledge distillation

**Audience:** DL-seminar peers who know neural nets & transformers, but not necessarily knowledge distillation.

**Goal of this notebook:** *teach the technique* behind the `patho-distill` project with a tiny, CPU-runnable toy you can read end-to-end in a few minutes — then connect it to the real pathology run.

We will:
1. Build several **frozen teachers** of *different embedding sizes* (mimicking UNI2 / Virchow2 / H-optimus).
2. Train **one small student** with a **projection head per teacher** to imitate all of them at once.
3. Use the **exact AM-RADIO loss** from the project (`distill/losses.py`).
4. Watch a **linear probe** on the frozen student get better as distillation proceeds — the same evaluation idea (`eva`) used for the real 70% → 83% BACH curve.

> Everything below runs on CPU in well under a minute. No GPU, no downloads, no labels for the distillation step.

In [ ]:
import sys, math
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = 'cpu'

# Reuse the PROJECT'S actual loss functions if the repo is importable.
sys.path.append('..')
try:
    from distill.losses import cosine_distance, summary_loss, feature_loss
    print('✓ Using the project\'s real loss from distill/losses.py')
except Exception as e:
    def cosine_distance(a, b):
        return (1 - F.cosine_similarity(a, b, dim=-1)).mean()
    def summary_loss(student, teacher):
        return cosine_distance(student, teacher)
    def feature_loss(student, teacher):
        return cosine_distance(student, teacher) + F.smooth_l1_loss(student, teacher)
    print('(repo not on path — using identical fallback definitions)')

## 1. A toy world: one hidden 'biology', many ways to see it

Real pathology tiles are unlabelled pixels. Each foundation model maps a tile to an *embedding*. Different models encode the same tissue **differently** and in **different dimensions**.

We simulate that: every sample has a hidden latent `z` ("the biology"). We render it into an *image vector* `x` (the "pixels" all models see). Each **teacher** is a *fixed, frozen* nonlinear network that maps `x` to its own embedding — a different network and a different size per teacher, just like the real ones.

In [ ]:
K       = 16     # hidden latent factors ('biology')
D_IMG   = 64     # 'pixel' / image-vector size every model sees
N_PATCH = 9      # toy spatial tokens (3x3) -- the real model uses 16x16 = 256

# Fixed renderer: latent -> image vector (the 'tile' all models receive)
R = torch.randn(K, D_IMG)
def render(z):
    return torch.tanh(z @ R)

def make_batch(n):
    z   = torch.randn(n, K)              # one global latent per sample
    zp  = torch.randn(n, N_PATCH, K)     # a latent per patch
    x_cls = render(z)                    # [n, D_IMG]      <- summary 'pixels'
    x_pat = render(zp)                   # [n, N_PATCH, D_IMG] <- patch 'pixels'
    return z, x_cls, x_pat

# --- Frozen teachers, named after the real ones, with the real (relative) dim gaps ---
TEACHER_DIMS = {'UNI2': 48, 'Virchow2': 32, 'H-optimus': 48}

def frozen_mlp(d_in, d_out, hidden=128, seed=0):
    g = torch.Generator().manual_seed(seed)
    net = nn.Sequential(nn.Linear(d_in, hidden), nn.GELU(), nn.Linear(hidden, d_out))
    with torch.no_grad():
        for p in net.parameters():
            p.copy_(torch.empty_like(p).normal_(0, 0.6, generator=g))
    for p in net.parameters():
        p.requires_grad_(False)
    return net.eval()

teachers = {name: frozen_mlp(D_IMG, d, seed=i+1) for i, (name, d) in enumerate(TEACHER_DIMS.items())}
print('Teachers (frozen):', {k: v[-1].out_features for k, v in teachers.items()})

In [ ]:
def standardize(x):
    """Zero-mean / unit-variance per channel (over batch & tokens) -- the AM-RADIO trick
    so teachers on different scales contribute comparably to the loss."""
    dims = tuple(range(x.dim() - 1))
    return (x - x.mean(dims, keepdim=True)) / (x.std(dims, keepdim=True) + 1e-6)

def teacher_targets(x_cls, x_pat):
    """Each teacher's standardized (summary, patch) embeddings -- the distillation targets."""
    out = {}
    for name, net in teachers.items():
        out[name] = (standardize(net(x_cls)), standardize(net(x_pat)))
    return out

## 2. The student: one shared backbone + a projection head per teacher

**Problem:** teachers output 48-d, 32-d, 48-d — the student only has a 768-d backbone (here a toy 24-d). You cannot compare a 24-d student vector to a 48-d teacher vector directly.

**Fix (AM-RADIO):** a small **MLP head per teacher** maps the shared backbone feature up to *that* teacher's size. The backbone is forced to learn a single *union* representation; each head reads off a teacher-specific view. The **same head** is applied to both the summary (CLS) token and every patch token — exactly as in `distill/models/student.py`.

In [ ]:
H = 24   # toy student backbone width (stands in for the real 768-d DINOv2 ViT-B)

class Student(nn.Module):
    def __init__(self, d_img, h, teacher_dims):
        super().__init__()
        self.backbone = nn.Sequential(nn.Linear(d_img, 128), nn.GELU(), nn.Linear(128, h))
        self.heads = nn.ModuleDict({
            name: nn.Sequential(nn.Linear(h, 128), nn.GELU(), nn.Linear(128, d))
            for name, d in teacher_dims.items()
        })
    def backbone_embed(self, x):
        return self.backbone(x)                 # the 'distilled' representation we evaluate
    def forward(self, x_cls, x_pat):
        h_cls = self.backbone(x_cls)
        h_pat = self.backbone(x_pat)
        proj = {}
        for name, head in self.heads.items():
            proj[name] = (head(h_cls), head(h_pat))  # shared head -> (summary, patches)
        return proj

student = Student(D_IMG, H, TEACHER_DIMS).to(device)
n_params = sum(p.numel() for p in student.parameters())
print(f'Student params: {n_params:,}  (backbone shared across all teachers)')

## 3. Distillation — the AM-RADIO loss

For each teacher we add a **summary loss** (cosine on the CLS token) and a **feature loss** (cosine + smooth-L1 on the patch tokens), then **sum over teachers**. This is literally:

```python
L = Σ_t  summary_loss(student_cls, teacher_cls) + feature_loss(student_pat, teacher_pat)
```

Note: **no labels are used here** — the target is the teacher's own embedding.

In [ ]:
opt = torch.optim.AdamW(student.parameters(), lr=2e-3, weight_decay=0.05)

STEPS, BATCH = 600, 128
hist = {'step': [], 'loss': [], **{f'cos_{n}': [] for n in teachers}}

for step in range(STEPS):
    _, x_cls, x_pat = make_batch(BATCH)
    tgt  = teacher_targets(x_cls, x_pat)
    proj = student(x_cls, x_pat)

    loss = 0.0
    for name in teachers:
        s_cls, s_pat = proj[name]
        t_cls, t_pat = tgt[name]
        loss = loss + summary_loss(s_cls, t_cls) + feature_loss(s_pat, t_pat)

    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)   # grad clip, as in the project
    opt.step()

    if step % 20 == 0 or step == STEPS - 1:
        with torch.no_grad():
            hist['step'].append(step); hist['loss'].append(float(loss))
            for name in teachers:
                s_cls, _ = proj[name]; t_cls, _ = tgt[name]
                hist[f'cos_{name}'].append(float(F.cosine_similarity(s_cls, t_cls, dim=-1).mean()))

print(f'final loss {hist["loss"][-1]:.3f}  |  CLS cosine-sim ' +
      ', '.join(f'{n}={hist[f"cos_{n}"][-1]:.2f}' for n in teachers))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(hist['step'], hist['loss'], color='#333'); ax[0].set_title('Total AM-RADIO loss')
ax[0].set_xlabel('step'); ax[0].set_ylabel('loss'); ax[0].grid(alpha=.3)
for name in teachers:
    ax[1].plot(hist['step'], hist[f'cos_{name}'], label=name)
ax[1].axhline(1.0, ls='--', c='gray', lw=1)
ax[1].set_title('Student→teacher CLS cosine similarity (↑1 is perfect)')
ax[1].set_xlabel('step'); ax[1].set_ylim(0, 1.02); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

**What you just saw:** a *single* shared backbone simultaneously climbs toward high cosine similarity with **all three** differently-sized teachers (in our run ≈ 0.88–0.91). That is the 'agglomerative' promise — one student, many experts. In the real run the same CLS cosine-sim sits around **0.76–0.85**.

### Quick ablation: why the per-teacher heads?
Without a head, the 24-d student feature cannot even be compared to a 48-d teacher. The heads are what make heterogeneous teachers compatible — try setting `H = 48` and removing a head to feel the dimension clash.

## 4. Does the student embedding become *useful*? — the linear-probe test

This is exactly how `eva` scores the real model: **freeze** the backbone, then train a tiny **linear classifier** on a labelled task. If a *linear* probe is accurate, the embedding has organised the information well.

We invent a 4-class label from the hidden latent `z` (think: 4 tissue subtypes, like **BACH**). The distillation never saw these labels. Our references are an **untrained student** (the floor) and **each individual teacher**. Watch the distilled student climb from the floor and overtake the teachers — because it fuses *all three* partial views into one.

In [ ]:
N_CLASS = 4
PROTO = torch.randn(K, N_CLASS)            # fixed class prototypes in latent space

def labelled_set(n):
    z, x_cls, _ = make_batch(n)
    y = (z @ PROTO).argmax(dim=1)          # 4-class label from the hidden biology
    return x_cls, y

def linear_probe_acc(embed_fn, n_tr=2000, n_te=1000, epochs=150):
    """Freeze the embedding, fit a linear classifier, report test accuracy (the eva idea)."""
    xtr, ytr = labelled_set(n_tr); xte, yte = labelled_set(n_te)
    with torch.no_grad():
        ftr, fte = embed_fn(xtr), embed_fn(xte)
    clf = nn.Linear(ftr.shape[1], N_CLASS)
    o = torch.optim.Adam(clf.parameters(), lr=1e-2)
    for _ in range(epochs):
        o.zero_grad(); F.cross_entropy(clf(ftr), ytr).backward(); o.step()
    with torch.no_grad():
        return float((clf(fte).argmax(1) == yte).float().mean())

# References: an UNTRAINED student (the floor) and each individual teacher.
untrained = Student(D_IMG, H, TEACHER_DIMS)
baselines = {'untrained student': linear_probe_acc(lambda x: untrained.backbone_embed(x))}
for name, net in teachers.items():
    baselines[name] = linear_probe_acc(lambda x, net=net: standardize(net(x)))
print('Reference linear-probe accuracy:', {k: round(v, 3) for k, v in baselines.items()})

In [ ]:
# Re-run distillation from scratch, probing the FROZEN student backbone along the way.
student = Student(D_IMG, H, TEACHER_DIMS)
opt = torch.optim.AdamW(student.parameters(), lr=2e-3, weight_decay=0.05)

curve_steps, curve_acc = [], []
for step in range(STEPS + 1):
    if step % 100 == 0:
        acc = linear_probe_acc(lambda x: student.backbone_embed(x))
        curve_steps.append(step); curve_acc.append(acc)
    _, x_cls, x_pat = make_batch(BATCH)
    tgt, proj = teacher_targets(x_cls, x_pat), student(x_cls, x_pat)
    loss = sum(summary_loss(proj[n][0], tgt[n][0]) + feature_loss(proj[n][1], tgt[n][1]) for n in teachers)
    opt.zero_grad(); loss.backward(); opt.step()

plt.figure(figsize=(7.5, 4.2))
plt.plot(curve_steps, curve_acc, marker='o', color='#006e87', lw=2, label='distilled student (frozen probe)')
for name, acc in baselines.items():
    plt.axhline(acc, ls='--', lw=1, alpha=.7, label=f'{name} ({acc:.2f})')
plt.title('Toy linear-probe learning curve (cf. real BACH 70% → 83%)')
plt.xlabel('distillation step'); plt.ylabel('probe accuracy'); plt.ylim(0.5, 0.95)
plt.legend(fontsize=8, ncol=2, loc='lower right'); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
print(f'untrained {curve_acc[0]:.2f}  →  distilled {curve_acc[-1]:.2f}   (best single teacher: {max(v for k,v in baselines.items() if k!="untrained student"):.2f})')

**The punchline:** with **no task labels in the loop**, the student's *frozen* features go from a useless random embedding to one that a linear classifier reads accurately — purely by **imitating the teachers**. And it ends up **above every individual teacher**: fusing three partial views into one stronger representation, while using a **single** small backbone at inference. That *student ≥ teachers* effect is the whole point of *agglomerative* distillation.

## 5. The real project

Same technique, real scale:

| | Toy (this notebook) | `patho-distill` (real) |
|---|---|---|
| Student | 24-d MLP | **DINOv2 ViT-B/14**, 768-d, 86.6 M |
| Teachers | 3 frozen MLPs (48/32/48-d) | **UNI2-h, Virchow2, H-optimus-1** (1536/1280/1536-d), 0.3–1.1 B each |
| Tokens | 1 + 9 | 1 CLS + **256** patches |
| Data | synthetic | **~248 GB** unlabelled tiles |
| Loss | `distill/losses.py` | *same* (cosine + smooth-L1) |
| Probe | toy 4-class | **BACH** 4-class via Kaiko `eva` |

**Real BACH linear-probe learning curve:** 70.1% @ step 1k → **83.3% ± 0.4** @ step 22k (AUROC 0.863 → 0.923) — +13 points in ~22% of training.

In [ ]:
real_steps = [1000, 22132]
real_acc   = [0.701, 0.833]
plt.figure(figsize=(6.5, 4))
plt.plot(real_steps, real_acc, marker='s', color='#006e87', lw=2)
for s, a in zip(real_steps, real_acc):
    plt.annotate(f'{a:.0%}', (s, a), textcoords='offset points', xytext=(0, 8), ha='center')
plt.title('Real run: BACH linear-probe accuracy vs. distillation step')
plt.xlabel('training step'); plt.ylabel('BACH accuracy'); plt.ylim(0.6, 0.9)
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## Take-home messages

- **Distillation compresses *knowledge*, not just weights** — one 86.6 M student replaces three 0.3–1.1 B teachers at inference.
- **Features, not labels** — matching embeddings makes distillation *label-free*, ideal when unlabelled data is abundant.
- **Heterogeneous teachers? Add a head each** — a shared backbone + per-teacher projection heads turns 'many experts' into one *union* representation.
- **Match direction *and* magnitude, global *and* local** — cosine + smooth-L1 on CLS + patch tokens copies the full geometry of the teacher.
- **The union can beat its parts** — one student fusing several teachers can out-probe any single teacher.

*See `handout.tex` for the 1-page companion. Loss code: `distill/losses.py`; student: `distill/models/student.py`; eval: Kaiko `eva`.*